# 线性回归的简洁实现

## 基于torch成熟框架的线性回归实现

In [1]:
import numpy as np
import torch
from torch.utils import data

In [2]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
num_examples = 1000
def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))
features, labels = synthetic_data(true_w, true_b, num_examples)

我们可以调用框架中现有的API来读取数据。 我们将features和labels作为API的参数传递，并通过数据迭代器指定batch_size。 此外，布尔值is_train表示是否希望数据迭代器对象在每个迭代周期内打乱数据。

In [3]:
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)#把每一行标签和每一行特征相对应
    return data.DataLoader(dataset, batch_size, shuffle=is_train)
#DataLoader：打乱顺序抽小批量样本
batch_size = 10
data_iter = load_array((features, labels), batch_size)

测试一下，获取第一项

In [4]:
next(iter(data_iter))

[tensor([[ 1.2462,  1.0770],
         [-0.8121,  0.7754],
         [-0.2795,  0.4202],
         [ 0.6688, -3.1669],
         [-1.5560, -2.1401],
         [ 1.3740, -0.3800],
         [-0.4112,  1.3553],
         [-1.1454,  0.9208],
         [-1.5076,  0.9296],
         [ 1.0521,  1.2768]]),
 tensor([[ 3.0569],
         [-0.0578],
         [ 2.2096],
         [16.3118],
         [ 8.3739],
         [ 8.2309],
         [-1.2404],
         [-1.2157],
         [-1.9844],
         [ 1.9654]])]

定义一个模型变量net，它是一个Sequential类的实例。 Sequential类将多个层串联在一起。 当给定输入数据时，Sequential实例将数据传入到第一层， 然后将第一层的输出作为第二层的输入，以此类推。 在下面的例子中，我们的模型只包含一个层，因此实际上不需要Sequential。 但是由于以后几乎所有的模型都是多层的，在这里使用Sequential会让你熟悉“标准的流水线”。至于linear代指线性，2，1分别指定输入与输出形状

In [5]:
# nn是神经网络的缩写
from torch import nn
net = nn.Sequential(nn.Linear(2, 1))

我们通过net[0]选择网络中的第一个图层， 然后使用weight.data和bias.data方法访问参数，net[0].weight 本身是一个带梯度追踪的参数对象，而 .data 取出它底层的张量数据，fill_表示当场填充

In [6]:
net[0].weight.data.normal_(0, 0.01)#把权重按正态分布概率取值，_表示直接更改
net[0].bias.data.fill_(0)

tensor([0.])

MSEloss求均方误差

In [7]:
loss = nn.MSELoss()

随机梯度下降优化器用 SGD 方法，以 0.03 的学习率，自动更新 net 里面的所有参数

In [8]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

在每个迭代周期里，我们将完整遍历一次数据集（train_data）， 不停地从中获取一个小批量的输入和相应的标签。 对于每一个小批量，我们会进行以下步骤:

通过调用net(X)生成预测并计算损失l（前向传播）。

通过进行反向传播来计算梯度。

通过调用优化器来更新模型参数。

In [13]:
num_epochs = 10
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000100
epoch 2, loss 0.000101
epoch 3, loss 0.000099
epoch 4, loss 0.000100
epoch 5, loss 0.000100
epoch 6, loss 0.000099
epoch 7, loss 0.000100
epoch 8, loss 0.000100
epoch 9, loss 0.000099
epoch 10, loss 0.000099


In [14]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([9.5141e-04, 9.5606e-05])
b的估计误差： tensor([-0.0002])
